# PLIQ v0.6 Tutorial

This notebook walks through **one pose** end-to-end and explains how **OTMol mapping** feeds **BINANA interaction recall** and the final **PLIQ score**.

**Prerequisites:** `pip install -e .` from the repo root; Open Babel (`obabel`) on PATH; OTMol installed.

**One-command run** (four PDBs → scored CSV):

```python
from pliq import run_pliq_from_pdbs
df = run_pliq_from_pdbs("ref_lig.pdb", "ref_pro.pdb", "dock_lig.pdb", "dock_pro.pdb", "pliq_result.csv")
print(df["pliq"].iloc[0])
```

Bundled example from the repo root: `python examples/7ZU2_DHT_model15/run_example.py`

Column guide: [`examples/7ZU2_DHT_model15/README.md`](../examples/7ZU2_DHT_model15/README.md). Scoring: [`scoring.md`](scoring.md).

## 1. Paths and bundled example (7ZU2 / DHT / model 15)

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()  # pliq_ver6 repo root when cwd=docs/
EX = ROOT / 'examples' / '7ZU2_DHT_model15'

REF_LIG = EX / 'ref' / '7ZU2_DHT_ligand_chainB.pdb'
REF_PRO = EX / 'ref' / '7ZU2_DHT_protein_renum.pdb'
DOCK_LIG = EX / 'dock' / 'posebusters_7ZU2_DHT_model_15_aligned_ligand.pdb'
DOCK_PRO = EX / 'dock' / 'posebusters_7ZU2_DHT_model_15_aligned_protein.pdb'

for p in (REF_LIG, REF_PRO, DOCK_LIG, DOCK_PRO):
    assert p.is_file(), p
print('Example PDBs OK')

## 2. Run full PLIQ (or load precomputed CSV)

Set `RUN_LIVE = True` to recompute (needs obabel, BINANA vendor, TM-score). Default loads bundled results.

In [ ]:
RUN_LIVE = False

if RUN_LIVE:
    from pliq import run_pliq_from_pdbs
    df = run_pliq_from_pdbs(
        REF_LIG, REF_PRO, DOCK_LIG, DOCK_PRO,
        EX / 'pliq_result_full_live.csv',
        pdb_id='7ZU2_DHT', af_model_id=15,
    )
else:
    df = pd.read_csv(EX / 'pliq_result_full.csv', nrows=1)

summary_cols = [
    'pdb_id', 'af_model_id', 'run_otmol_ok', 'run_binana_ok', 'run_posebusters_ok', 'run_tmscore_ok',
    'ligand_rmsd', 'otmol_BCI', 'otmol_bci_zero_ok', 'pliq_mapping_trusted',
    'interface_rmsd', 'fnat_all', 'binana_hbond_tp_n', 'binana_hydrophobic_tp_n',
    'tm_TMscore', 'pb_valid_fraction', 'pliq',
    'pliq_term_posebusters', 'pliq_term_B6xH', 'pliq_term_kernel_i', 'pliq_term_kernel_l', 'pliq_term_TM',
]
summary_cols = [c for c in summary_cols if c in df.columns]
df[summary_cols].T

## 3. OTMol mapping — once per pose

- **`use_original_coords_rmsd=True`** → `ligand_rmsd` is on **original PDB coordinates** (no ligand superposition).
- **`reflection=False`** always.
- Output: `dock_to_ref` (dock RDKit idx → ref RDKit idx), `otmol_BCI`, `ligand_rmsd`.

In [ ]:
row = df.iloc[0]
print(f"ligand_rmsd (OTMol, original coordinates): {float(row['ligand_rmsd']):.4f} Å")
print(f"BCI: {float(row['otmol_BCI'])}  trusted: {row['pliq_mapping_trusted']}")
pairs = str(row.get('otmol_map_pairs', row.get('ligand_dock_to_ref_pairs', '')))
print(f"dock_to_ref pairs: {pairs[:80]}...")

## 4. BINANA × OTMol — ligand signatures in reference frame

BINANA runs on ref and dock separately. For each interaction:

1. Map BINANA `atomIndex` → RDKit idx (by 3D coordinates).
2. Dock ligand atoms: `dock RDKit → dock_to_ref → ref RDKit`.
3. Build signature `L[ref_rdkit_indices]R[receptor_part]`.
4. Multiset TP/FN/FP between ref and dock signatures.

**BINANA does not call OTMol again** — it reuses the mapping from step 3.

In [ ]:
row = df.iloc[0]
print('BINANA recall reuses the OTMol dock_to_ref map (no second align)')
for short in ['hbond', 'hydrophobic', 'salt', 'pipi']:
    tp = row.get(f'binana_{short}_tp_n', 'NA')
    f1 = row.get(f'binana_{short}_default_f1', row.get(f'binana_{short}_f1', 'NA'))
    print(f"{short:12s}  TP={tp}  F1={f1}")

sig = str(row.get('binana_hbond_tp_sig_list', ''))[:120]
if sig:
    print('\nExample hbond TP signature:', sig)

## 5. PLIQ score decomposition

```
pliq = term_4 × (B6×H + k_i + k_l) / 3 × TM
term_4 = (pb_valid_fraction)²
k_i = 1/(1+(interface_rmsd/1.5)²)
k_l = 1/(1+(ligand_rmsd/8.5)²)
```

In [ ]:
r = df.iloc[0]

terms = [
    'pliq_term_posebusters', 'pliq_term_B6', 'pliq_term_H', 'pliq_term_B6xH',
    'pliq_term_kernel_i', 'pliq_term_kernel_l', 'pliq_term_TM', 'pliq',
]
for t in terms:
    if t in df.columns:
        print(f"{t:24s} {float(r[t]):.4f}")

t4 = float(r['pliq_term_posebusters'])
bracket = (float(r['pliq_term_B6xH']) + float(r['pliq_term_kernel_i']) + float(r['pliq_term_kernel_l'])) / 3
tm = float(r['pliq_term_TM'])
reconstructed = t4 * bracket * tm
print(f"\nReconstructed pliq: {reconstructed:.4f}  (stored: {float(r['pliq']):.4f})")

## 6. Publication checklist

Run from repo root:

```bash
pytest tests/ -v
python scripts/verify_pliq_publication.py
python examples/7ZU2_DHT_model15/run_example.py
```

For benchmark analysis, typical filters:
- `otmol_BCI == 0`
- `pliq_mapping_trusted == True`
- optional BINANA sequence-consistency audit (see analysis scripts)